# Belief Propagation Decoding for CSS qLDPC Codes

Belief propagation (BP) uses an observed syndrome and a noise model to estimate the probabilities of individual error bits. The noise model assigns *prior probabilities* to possible errors, which can be updated by conditioning on the observed syndrome to obtain a *posterior distribution*. Of particular interest are the *posterior marginal probabilities* of individual error bits, obtained by summing over the possible values of the remaining bits. BP estimates these marginals by exchanging probabilistic information between data nodes representing error bits and check nodes representing parity-check constraints on a Tanner graph. These estimates are called *beliefs*.

For Tanner graphs containing cycles, as in our qLDPC code, these estimates are generally approximate [1]. Furthermore, independently choosing the most likely value of each error bit need not produce an estimate satisfying the observed syndrome. We implement sum-product BP in log-likelihood-ratio (LLR) form, following Higgott and Breuckmann [1] and using Ginsberg [2] as an implementation reference. Here we develop and validate BP before introducing ordered-statistics decoding (OSD) as postprocessing in a subsequent notebook [1].

## 1. Noise Model and Probabilistic Decoding

### Prior Error Probabilities

We consider independent depolarizing noise on the $n$ physical qubits of a CSS code under a *code-capacity noise model*: Pauli errors affect the data qubits, while syndrome measurements are assumed perfect. This model provides both the error distributions used in our simulations and the prior probabilities required by BP. Each qubit independently experiences a Pauli error according to

$$
P(I)=1-p \quad \text{ and } \quad P(X)=P(Y)=P(Z)=\frac{p}{3},
$$

where $p\in[0,1]$ is the physical error probability.

As in the preceding notebook, a Pauli error is represented by two binary vectors $e_X,e_Z\in\mathbb F_2^n$. We first determine the marginal probability distributions of these components, which will provide the prior information used by our BP decoder. For each $0\leq i<n$, the binary random variables $e_X[i]$ and $e_Z[i]$ take values in $\{0,1\}$. Since $e_X[i]=1$ corresponds to either an $X$ or a $Y$ error, we have

$$P(e_X[i]=1)=P(X)+P(Y)=\frac{2p}{3},$$

and, by a similar argument,

$$P(e_Z[i]=1)=P(Z)+P(Y)=\frac{2p}{3}.$$

Thus, both binary error components have the same marginal distribution:

$$
e_X[i],e_Z[i]\sim\operatorname{Bernoulli}(q)
\quad\text{for}\quad q=\frac{2p}{3}.
$$

Each component can therefore be modeled as the error introduced by a binary symmetric channel with crossover probability $q$. However, the $X$ and $Z$ components on a given qubit are generally correlated: $P(e_X[i]=1,e_Z[i]=1)=\frac p3$, whereas $P(e_X[i]=1)P(e_Z[i]=1)=\frac{4p^2}{9}$. These probabilities agree only when $p=0$ or $p=3/4$. Our BP implementation nevertheless decodes the two components separately using their marginal distributions as priors, without exploiting these correlations [3, Sec. II.E, Eqs. (43)–(47)].


### From Prior Probabilities to Posterior Marginals

Let $e\in\mathbb F_2^n$ denote either binary error component, with corresponding parity-check matrix $H$ and observed syndrome $s$. Since the equation $He=s$ generally has multiple solutions, the syndrome alone does not uniquely determine the error. We therefore combine the syndrome information with prior probabilities determined by the noise model.

Because error bits within each component are independent across qubits, the prior probability of an error vector is

$$P(e)=q^{\operatorname{wt}(e)}(1-q)^{n-\operatorname{wt}(e)},$$

where $\operatorname{wt}(e)$ is the Hamming weight of $e$. Assuming perfect syndrome measurements, Bayes' rule gives

$$P(e\mid s)=\frac{P(s\mid e)P(e)}{P(s)} \propto P(e)\,\mathbf{1}\{He=s\},$$

where $\mathbf{1}\{He=s\}$ equals 1 when the syndrome equation is satisfied and 0 otherwise. Thus, errors inconsistent with the syndrome have conditional probability zero, while the relative probabilities of consistent errors are determined by their priors.

Sum-product BP estimates the **posterior marginal probabilities** of individual error bits, rather than directly finding the most probable complete error vector [1, Sec. V, Eq. (13)]:

$$P(e[i]=a\mid s)=\sum_{\substack{e'\in\mathbb F_2^n\\e'[i]=a}}P(e'\mid s),\qquad a\in\{0,1\}.$$

Thus, sum-product BP exploits the factorization of the prior distribution and syndrome constraints to estimate these marginals through message passing on the Tanner graph, without enumerating all possible errors. The estimates are exact on cycle-free graphs but generally approximate on graphs containing cycles.

### Initial Log-Likelihood Ratios

To implement BP, we represent the probabilistic information exchanged between nodes using **log-likelihood ratios** (LLRs). For a binary error variable $e$, define

$$L=\ln\frac{P(e=0)}{P(e=1)}.$$

A positive LLR favors $e=0$, while a negative LLR favors $e=1$. Its absolute value measures the strength of that preference.

Using $q=2p/3$ from the noise model, the initial LLR for each binary error variable is

$$L_i^{(0)}=\ln\frac{1-q}{q}.$$

Since the noise model assigns the same marginal error probability to every qubit, all binary error variables have the same initial LLR.

In [1]:
from itertools import product

import jax.numpy as jnp
import numpy as np

from qldpc_bposd_jax.bp import (
    binary_prior_llr,
    bp_iteration,
    check_to_data_update,
    data_to_check_update,
    decode_binary,
    hard_decision,
    posterior_llrs,
    syndrome_matches,
)
from qldpc_bposd_jax.codes import bb72_check_matrices
from qldpc_bposd_jax.logical import is_in_row_space, logical_success
from qldpc_bposd_jax.noise import sample_error_and_syndrome
from qldpc_bposd_jax.syndrome import binary_syndrome

In [2]:
p = 0.06
L0 = binary_prior_llr(p)

print(f"Physical error probability: {p}")
print(f"Binary error probability: {2 * p / 3:.4f}")
print(f"Initial LLR: {float(L0):.4f}")

Physical error probability: 0.06
Binary error probability: 0.0400
Initial LLR: 3.1781



## 2. Tanner Graph Representation

We use the Tanner graph associated with a binary parity-check matrix $H\in\mathbb F_2^{m\times n},$ as introduced in the preceding notebook.

To implement BP, we represent the graph by an edge list, assigning an index to each nonzero entry of $H$. Each edge connects a data node to a check node and carries two messages: one in each direction. We first construct this representation for a small parity-check matrix before applying it to the $[[72,12,6]]$ BB code.



### Edge-List Representation

Consider the parity-check matrix

$$
H=
\begin{pmatrix}
1&1&0&0\\
0&1&1&1
\end{pmatrix}.
$$

We represent its Tanner graph using two arrays, `check_idx` and `data_idx`, whose corresponding entries specify the endpoints of each edge. Thus, for edge $k$, the pair `(check_idx[k], data_idx[k])` specifies the indices $j$ and $i$ of its endpoints $c_j$ and $d_i$, respectively. This representation will allow us to associate data-to-check and check-to-data messages with individual edges. 

In [3]:
H_demo = np.array(
    [
        [1, 1, 0, 0],
        [0, 1, 1, 1],
    ],
    dtype=np.uint8,
)

# since H_demo is 2D array, np.nonzero returns two arrays:
# the first describing the row indices and the second corresponding to the column indices of nonzero entries in H_demo
check_idx, data_idx = np.nonzero(H_demo)

# store edge endpoints as JAX arrays.
check_idx = jnp.asarray(check_idx)
data_idx = jnp.asarray(data_idx)

# the k-th entry of each array identifies the two endpoints of edge k
print("Check indices:", check_idx)
print("Data indices:", data_idx)
print("Number of edges:", len(check_idx))

Check indices: [0 0 1 1 1]
Data indices: [0 1 1 2 3]
Number of edges: 5


## 3. BP Message Updates

Each Tanner-graph edge carries two LLR messages, one in each direction, denoted $L_{d_i\to c_j}^{(t)}$ and $L_{c_j\to d_i}^{(t)}$, where $t$ indicates the iteration number. Each message expresses a preference for one value of an error bit over the other, with its sign indicating the favored value and its magnitude indicating the strength of that preference.

We initialize data-to-check messages using the prior LLRs derived in Section 1 and check-to-data messages to zero. We then use a *flooding schedule*: each iteration consists of parallel check-to-data updates followed by parallel data-to-check updates, corresponding to the horizontal and vertical steps described by Higgott and Breuckmann [1, Sec. V]. Check nodes use the observed syndrome and incoming data-to-check messages to compute their outgoing messages. Data nodes then combine the newly received check-to-data messages with their prior LLRs to compute the outgoing data-to-check messages.

After each iteration, we compute estimates of the posterior LLRs for each error bit, called *soft decisions*. Their signs determine *hard decisions* for the individual error bits, producing a candidate error vector that we test for consistency with the observed syndrome. This process repeats until the candidate reproduces the syndrome or the maximum number of iterations is reached.

### Message Initialization

The case $t=0$ denotes initialization, before any message updates have occurred. As described above, we initialize each data-to-check message with the prior LLR of its data node and each check-to-data message to zero:

$$L_{d_i\to c_j}^{(0)}=L_i^{(0)} \quad \text{ and } \quad L_{c_j\to d_i}^{(0)}=0.$$

The check-to-data messages are initialized to zero because no information from the parity checks has yet been propagated. The first check-to-data update occurs during iteration $t=1$, and uses the initialized data-to-check messages and the observed syndrome.

In [4]:
prior_llrs_demo = jnp.full(H_demo.shape[1], L0)

# initialize one message in each direction per edge
data_to_check = prior_llrs_demo[data_idx]
check_to_data = jnp.zeros_like(data_to_check)

print("Data-to-check:", data_to_check)
print("Check-to-data:", check_to_data)

Data-to-check: [3.1780539 3.1780539 3.1780539 3.1780539 3.1780539]
Check-to-data: [0. 0. 0. 0. 0.]



### Check-to-Data Messages

During the first phase of each BP iteration, every check node uses its parity-check equation and incoming data-to-check messages to compute an outgoing message for each neighboring data node.

Recall that the parity-check constraint associated with check node $c_j$ is

$$\sum_{i\in N_H(c_j)}e[i]=s[j]\pmod 2.$$

The message from $c_j$ to $d_i$ expresses how strongly this constraint favors $e[i]=0$ over $e[i]=1$, given the observed syndrome bit $s[j]$ and the incoming messages from the other neighboring data nodes.

We use the sum-product **check-to-data update** in LLR form [1, Eq. (14)]:
$$
L_{c_j\to d_i}^{(t)}
=
2\operatorname{arctanh}\left(
(-1)^{s[j]}
\prod_{i'\in N_H(c_j)\setminus\{i\}}
\tanh\left(
\frac{L_{d_{i'}\to c_j}^{(t-1)}}{2}
\right)
\right).
$$

Here $N_H(c_j)=\{i:H_{ji}=1\}$ denotes the neighboring data nodes of $c_j$. The product combines incoming information from the other neighboring data nodes according to the parity constraint, while the factor $(-1)^{s[j]}$ incorporates the observed syndrome. Excluding $d_i$ from the product prevents its incoming message from being immediately returned to it. Expressing the incoming LLRs through $\tanh(L/2)$ allows the parity constraint to be evaluated as a product of contributions from neighboring data nodes. Applying $2\operatorname{arctanh}$ then converts the result back into LLR form.

We implement this update using the edge-list representation introduced in Section 2.


#### Implementation

The function `check_to_data_update` in `bp.py` computes all check-to-data messages using the edge-list representation.

For each outgoing edge $k$, we construct a Boolean mask that selects the other edges incident to the same check node. We apply `tanh` to half of each incoming LLR, replace excluded factors with the multiplicative identity `1.0`, and compute the product along each row of the mask. We then incorporate the syndrome-dependent sign and apply `arctanh` to obtain the outgoing LLRs.

Following the numerical safeguard used by Ginsberg [2, *BP in JAX*], we clip the argument of `jnp.arctanh` to lie strictly within $(-1,1)$. This prevents infinite LLRs caused by floating-point saturation, at the cost of limiting the maximum message magnitude [1, Appendix A]. With our float32 clipping threshold, outgoing LLR magnitudes are bounded by approximately $16.64$, an approximation whose effect may need to be assessed during benchmarking.

#### Example
For our demonstration matrix $H_{\mathrm{demo}}$, consider the first BP iteration with physical error probability $p=0.06$ and observed syndrome $s=(1,0)$. In this case, the initialized data-to-check messages all have LLR $L^{(0)}\approx 3.1781$.

For the message from $c_0$ to $d_0$, the only contributing incoming message is from $d_1$. Since $s[0]=1$, the check-to-data update gives

$$
L_{c_0\to d_0}^{(1)}=-L^{(0)}\approx-3.1781.
$$

The negative LLR indicates that the parity-check constraint favors $e[0]=1$.

For the message from $c_1$ to $d_1$, the contributing incoming messages are from $d_2$ and $d_3$. Since $s[1]=0$, we obtain

$$
L_{c_1\to d_1}^{(1)}
=
2\operatorname{arctanh}\left(
\tanh\left(\frac{L^{(0)}}{2}\right)^2
\right)
\approx 2.4866.
$$

This positive LLR favors $e[1]=0$. Its magnitude is smaller because the parity of two uncertain error bits is less certain than either bit individually.

In [5]:
s_demo = jnp.array([1, 0], dtype=jnp.uint8)
check_to_data = check_to_data_update(data_to_check, check_idx, s_demo)
print("First-iteration check-to-data LLRs:", check_to_data)

First-iteration check-to-data LLRs: [-3.178054  -3.178054   2.4866414  2.4866414  2.4866414]


### Data-to-Check Messages

During the second phase of each BP iteration, each data node combines its prior LLR with incoming check-to-data messages, excluding the message from the destination check node. The resulting update is [1, Eq. (15)]:

$$
L_{d_i\to c_j}^{(t)}
=
L_i^{(0)}
+
\sum_{j'\in N_H(d_i)\setminus\{j\}}
L_{c_{j'}\to d_i}^{(t)}.
$$

Here $N_H(d_i)$ denotes the neighboring check nodes of $d_i$. Note in particular that the syndrome dependence is already contained in the incoming check-to-data messages.


#### Implementation

The function `data_to_check_update` in `bp.py` accumulates incoming check-to-data messages at each data node using `jnp.zeros_like(...).at[data_idx].add(...)`. It then gathers these sums by edge, adds the corresponding prior LLR, and subtracts the incoming message along that edge. This implements the exclusion of the receiving edge without a Python loop.

#### Example

Building off of our running example, observe that edges 1 and 2 are both incident to $d_1$. Each outgoing data-to-check message includes the prior LLR of $d_1$ and the incoming message from the other edge. The following calculation illustrates this exchange of messages.

In [6]:
data_to_check_next = data_to_check_update(prior_llrs_demo, check_to_data, data_idx)
print("Updated data-to-check LLRs:", data_to_check_next)

Updated data-to-check LLRs: [ 3.1780539e+00  5.6646953e+00 -2.3841858e-07  3.1780539e+00
  3.1780539e+00]



### Posterior LLRs and Hard Decisions

After each check-to-data update, BP estimates the posterior LLR of each error bit by combining its prior LLR with all incoming check-to-data messages [1, Eq. (16)]:

$$
L_i^{(t)}
=
L_i^{(0)}
+
\sum_{j\in N_H(d_i)}
L_{c_j\to d_i}^{(t)}.
$$

Unlike data-to-check messages, these estimates include contributions from every neighboring check node. They are called BP's soft decisions and provide the reliability information used by OSD [1, Sec. VI].

In [7]:
posterior = posterior_llrs(prior_llrs_demo, check_to_data, data_idx)
print("Posterior LLRs:", posterior)

Posterior LLRs: [-2.3841858e-07  2.4866412e+00  5.6646953e+00  5.6646953e+00]



Having computed the posterior LLR estimates, we use them to construct a candidate error vector. Recall that, at iteration $t$, BP estimates the log-likelihood ratio

$$L_i^{(t)} \approx \ln\frac{P(e[i]=0\mid s)}{P(e[i]=1\mid s)}.$$

We obtain an estimated error vector $\hat e^{(t)}$ by choosing the more probable value of each error bit according to its posterior LLR. This gives the hard decision rule [1, Sec. V]:

$$
\hat e^{(t)}[i]
=
\begin{cases}
0,&L_i^{(t)}\geq 0,\\
1,&L_i^{(t)}<0.
\end{cases}
$$

Note that we resolve ties at $L_i^{(t)}=0$ in favor of $0$, following Ginsberg [2], whereas Higgott and Breuckmann [1, Sec. V] choose 1.

Since the hard decisions are made independently, the resulting estimate need not satisfy $H\hat e^{(t)}=s$. We therefore test syndrome consistency after each iteration and stop when it holds or the maximum iteration count is reached.


We now examine the error estimate obtained from the posterior LLRs after the first BP iteration and check whether it reproduces the observed syndrome.

In [8]:
estimate = hard_decision(posterior)

print("Estimated error:", estimate)
print("Estimated syndrome:", binary_syndrome(H_demo, estimate))
print("Observed syndrome:", s_demo)
print("Syndrome consistent:", syndrome_matches(H_demo, estimate, s_demo))

Estimated error: [1 0 0 0]
Estimated syndrome: [1 0]
Observed syndrome: [1 0]
Syndrome consistent: True


After the first iteration, the posterior LLR for $d_0$ is theoretically zero. In our calculation, floating-point rounding makes it slightly negative, causing the hard-decision rule to select 1. The resulting estimate is syndrome-consistent, illustrating how numerical precision can affect decisions near an LLR of zero. However, the posterior LLRs have not yet reached their exact values. As we will examine in Section 5, satisfying the syndrome equations does not imply that the probabilistic estimates have converged.

## 4. Iterative BP Decoding

We combine the message updates and hard-decision rule into an iterative decoder using the functions implemented in `bp.py`. The function `bp_iteration` performs one iteration of the flooding schedule, returning the updated messages, posterior LLR estimates, and candidate error vector. The function `bp_decode` repeats these iterations until the estimated error reproduces the observed syndrome or the maximum iteration count is reached, following the stopping criterion described by Higgott and Breuckmann [1, Sec. V].

To make the decoder easier to use, we provide a separate function, `decode_binary`, which constructs the Tanner-graph edge list, initializes the prior LLRs from the physical error probability, and calls `bp_decode`. This function also handles the noiseless case $p=0$ without running BP.

The decoder returns the estimated error vector, a Boolean indicating syndrome consistency, and the number of iterations performed. We emphasize that syndrome consistency does not guarantee successful logical decoding, since an estimated error may differ from the actual error by a nontrivial logical operator.

In [9]:
estimate, consistent, iterations = decode_binary(H_demo, s_demo, p=p, max_iter=10)

print("Estimated error:", estimate)
print("Syndrome consistent:", consistent)
print("Iterations:", iterations)

Estimated error: [1 0 0 0]
Syndrome consistent: True
Iterations: 1


A syndrome-consistent estimate is sufficient for our BP stopping criterion, but it does not guarantee that the posterior marginals have been computed accurately. We next test the latter independently by comparing BP against exact posterior probabilities on our cycle-free demonstration graph.

## 5. Validation Against Exact Posterior Marginals

We validate the BP implementation by comparing its posterior LLR estimates with exact values obtained by enumerating all $2^4=16$ binary error vectors for the demonstration matrix.

Under perfect syndrome measurements, Bayes' rule gives the conditional probability of a syndrome-consistent error vector as

$$
P(e\mid s)
=
\frac{P(e)}
{\displaystyle\sum_{e':\,He'=s}P(e')}.
$$

The denominator is the probability of observing the syndrome, obtained via the law of total probability by summing the prior probabilities of all syndrome-consistent error vectors. Errors that do not satisfy $He=s$ have conditional probability zero. In the following cell, we enumerate all syndrome-consistent errors, normalize their prior probabilities using this expression, and calculate the exact posterior marginals and LLRs.

Since the demonstration Tanner graph is cycle-free, sum-product BP computes exact posterior marginals after sufficient message passing [1, Sec. V]. In this example, two iterations are sufficient for information to propagate between all data nodes. We therefore run exactly two iterations and compare the resulting LLRs with those obtained by exhaustive enumeration. We use `bp_iteration` rather than `decode_binary` for this comparison because the latter may stop as soon as a syndrome-consistent estimate is found, before the posterior LLRs have reached their exact values. We compare the posterior LLRs using an absolute tolerance of $10^{-5}$ to accommodate floating-point rounding in the JAX calculations.

In [10]:
# enumerate errors and select those consistent with the syndrome
n = H_demo.shape[1]
all_errors = jnp.array(list(product((0, 1), repeat=n)), dtype=jnp.uint8)

matches = jnp.all((all_errors @ H_demo.T) % 2 == s_demo, axis=1)
consistent_errors = all_errors[matches]

# calculate the conditional error probabilities
q = 2 * p / 3
weights = jnp.sum(consistent_errors, axis=1)
prior_probs = q**weights * (1 - q) ** (n - weights)
conditional_probs = prior_probs / jnp.sum(prior_probs)

# calculate exact posterior marginals and LLRs
exact_marginals = conditional_probs @ consistent_errors
exact_llrs = jnp.log1p(-exact_marginals) - jnp.log(exact_marginals)

# initialize data-to-check messages
data_to_check = prior_llrs_demo[data_idx]

# perform two BP iterations
for _ in range(2):
    data_to_check, _, bp_llrs, _ = bp_iteration(
        prior_llrs_demo, data_to_check, check_idx, data_idx, s_demo
    )

assert jnp.allclose(bp_llrs, exact_llrs, atol=1e-5, rtol=0)

print("Exact posterior LLRs:", exact_llrs)
print("BP posterior LLRs:   ", bp_llrs)
print("Maximum absolute error:", jnp.max(jnp.abs(bp_llrs - exact_llrs)))

Exact posterior LLRs: [-2.4866414  2.4866414  3.1780539  3.1780539]
BP posterior LLRs:    [-2.4866476  2.4866412  3.1780536  3.1780536]
Maximum absolute error: 6.198883e-06


## 6. BP Decoding for the $[[72,12,6]]$ Bivariate Bicycle Code

Having validated our BP implementation on a small cycle-free Tanner graph, we now apply it to the $[[72,12,6]]$ bivariate bicycle code introduced by Bravyi et al. [4].

On a cycle-free Tanner graph, sum-product BP computes exact posterior marginals after sufficient message passing. On a graph containing cycles, however, messages can eventually propagate back to nodes that contributed to them earlier. Consequently, the messages are no longer guaranteed to represent independent information. BP can effectively count correlated evidence multiple times, potentially leading to inaccurate beliefs, oscillation, or failure to find a syndrome-consistent estimate [1, Sec. V]. Unlike our demonstration graph, the Tanner graphs associated with the BB code contain cycles. We therefore no longer have a guarantee that BP will compute exact posterior marginals or converge to a syndrome-consistent estimate.

Recall that the CSS syndrome equations are

$$s_X=H_Xe_Z \quad \text{ and } \quad s_Z=H_Ze_X.$$

We decode the two error components separately, using $H_Z$ to estimate $e_X$ and $H_X$ to estimate $e_Z$. We begin with single-qubit Pauli errors before considering errors sampled from the depolarizing noise model.


### BP Decoding on Single-Qubit Pauli Errors

We import the BB parity-check matrices using `bb72_check_matrices()` from `codes.py` and apply `decode_binary` to single-qubit $X_1$, $Z_1$, and $Y_1$ errors on the first physical qubit (index 0 in Python).

For the BB-code experiments, we use a maximum of 20 BP iterations per error component, terminating earlier when an estimate satisfies the observed syndrome. This is an initial computational limit rather than a theoretically justified convergence threshold. We will examine the effect of the iteration budget during subsequent benchmarking.

In [11]:
H_X, H_Z = bb72_check_matrices()

print("H_X shape:", H_X.shape)
print("H_Z shape:", H_Z.shape)

H_X shape: (36, 72)
H_Z shape: (36, 72)


In [12]:
# introduce an X error on physical qubit 0
e_X = jnp.zeros(72, dtype=jnp.uint8).at[0].set(1)

# compute the observed Z-check syndrome
s_Z = binary_syndrome(H_Z, e_X)

# decode the X-error component
estimate_X, consistent_X, iterations_X = decode_binary(H_Z, s_Z, p=0.06, max_iter=20)

print("Actual X error:", e_X)
print("Estimated X error:", estimate_X)
print("Observed syndrome weight:", int(jnp.sum(s_Z)))
print("Syndrome consistent:", consistent_X)
print("BP iterations:", iterations_X)
print("Exact error recovery:", bool(jnp.array_equal(estimate_X, e_X)))

Actual X error: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Estimated X error: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Observed syndrome weight: 3
Syndrome consistent: True
BP iterations: 1
Exact error recovery: True


For this single-qubit X error, BP recovers the exact physical error after one iteration. The observed syndrome has Hamming weight three, consistent with the column weight of $H_Z$. Since the estimated and actual errors agree, the residual error is zero, so decoding is also logically successful in this example.

We now repeat the experiment for the $Z$ component, using $H_X$ instead of $H_Z$.

In [13]:
e_Z = jnp.zeros(72, dtype=jnp.uint8).at[0].set(1)

s_X = binary_syndrome(H_X, e_Z)

estimate_Z, consistent_Z, iterations_Z = decode_binary(H_X, s_X, p=0.06, max_iter=20)

print("Actual Z error:", e_Z)
print("Estimated Z error:", estimate_Z)
print("Observed syndrome weight:", int(jnp.sum(s_X)))
print("Syndrome consistent:", consistent_Z)
print("BP iterations:", iterations_Z)
print("Exact error recovery:", bool(jnp.array_equal(estimate_Z, e_Z)))

Actual Z error: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Estimated Z error: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Observed syndrome weight: 3
Syndrome consistent: True
BP iterations: 1
Exact error recovery: True


BP also recovers the single-qubit $Z$ error after one iteration. We next consider a $Y$ error, which has both $X$ and $Z$ components and therefore requires both binary decoders.

In [14]:
# a single-qubit Y error has both X and Z components
e_Y = jnp.zeros(72, dtype=jnp.uint8).at[0].set(1)

s_X_Y = binary_syndrome(H_X, e_Y)
s_Z_Y = binary_syndrome(H_Z, e_Y)

# decode the X component using H_Z
estimate_X_Y, consistent_X_Y, iterations_X_Y = decode_binary(
    H_Z, s_Z_Y, p=0.06, max_iter=20
)

# decode the Z component using H_X
estimate_Z_Y, consistent_Z_Y, iterations_Z_Y = decode_binary(
    H_X, s_X_Y, p=0.06, max_iter=20
)

print("X-check syndrome weight:", int(jnp.sum(s_X_Y)))
print("Z-check syndrome weight:", int(jnp.sum(s_Z_Y)))
print("Total syndrome weight:", int(jnp.sum(s_X_Y) + jnp.sum(s_Z_Y)))

print("X component recovered:", bool(jnp.array_equal(estimate_X_Y, e_Y)))
print("Z component recovered:", bool(jnp.array_equal(estimate_Z_Y, e_Y)))

print("X component syndrome consistent:", consistent_X_Y)
print("Z component syndrome consistent:", consistent_Z_Y)

print("X component BP iterations:", iterations_X_Y)
print("Z component BP iterations:", iterations_Z_Y)

X-check syndrome weight: 3
Z-check syndrome weight: 3
Total syndrome weight: 6
X component recovered: True
Z component recovered: True
X component syndrome consistent: True
Z component syndrome consistent: True
X component BP iterations: 1
Z component BP iterations: 1


For each of the single-qubit X, Z, and Y errors considered above, BP recovers the exact physical error after one iteration. The X and Z errors each produce a syndrome of Hamming weight three, while the Y error produces three nonzero bits in each syndrome component. In all three cases, the residual error is zero, so decoding is also logically successful.

These examples establish that our BP implementation works for these simple errors on the BB code, but they do not yet tell us how it behaves under more general Pauli noise.

### BP Decoding Under Sampled Pauli Noise

Having considered individual X, Z, and Y errors, we now examine BP decoding for an error sampled from the depolarizing noise model. We use `sample_error_and_syndrome` from `noise.py` to generate the binary error components and their corresponding CSS syndromes, then apply the binary BP decoder separately to each component. 

In [15]:
# fixed random seed for reproducibility
rng = np.random.default_rng(137)
p = 0.06

# sample a Pauli error and its CSS syndrome
e_X, e_Z, s_X, s_Z = sample_error_and_syndrome(H_X, H_Z, p, rng)

# decode the X component using the Z-check matrix
estimate_X, consistent_X, iterations_X = decode_binary(H_Z, s_Z, p=p, max_iter=20)

# decode the Z component using the X-check matrix
estimate_Z, consistent_Z, iterations_Z = decode_binary(H_X, s_X, p=p, max_iter=20)

print("Pauli error weight:", int(np.count_nonzero(e_X | e_Z)))
print("X component syndrome consistent:", consistent_X)
print("Z component syndrome consistent:", consistent_Z)
print("X component BP iterations:", iterations_X)
print("Z component BP iterations:", iterations_Z)

exact_recovery = np.array_equal(np.asarray(estimate_X), e_X) and np.array_equal(
    np.asarray(estimate_Z), e_Z
)

print("Exact Pauli recovery:", exact_recovery)

Pauli error weight: 4
X component syndrome consistent: True
Z component syndrome consistent: True
X component BP iterations: 1
Z component BP iterations: 3
Exact Pauli recovery: True


In [16]:
success = logical_success(H_X, H_Z, e_X, e_Z, estimate_X, estimate_Z)

print("Logical decoding success:", success)

Logical decoding success: True



### BP Decoding Across Multiple Error Samples

Our preceding experiment considered a single Pauli error sampled from the depolarizing noise model. We now extend this calculation to a small collection of independently sampled errors at a fixed physical error probability $p=0.06$. Using a fixed random seed makes the experiment reproducible.

For each sample, we will record the physical Pauli error weight, whether BP finds syndrome-consistent estimates for both error components, the number of iterations required, and whether decoding is successful. We will distinguish exact recovery of the physical error from logical decoding success, which only requires the residual error to be a stabilizer. This small experiment is intended to illustrate BP's behavior on the bivariate bicycle code rather than provide a quantitative assessment of its decoding performance.

In [17]:
p_trials = 0.06
num_trials = 15
rng_trials = np.random.default_rng(2026)

samples = [
    sample_error_and_syndrome(H_X, H_Z, p_trials, rng_trials) for _ in range(num_trials)
]

weights = [
    int(np.count_nonzero(e_X_trial | e_Z_trial))
    for e_X_trial, e_Z_trial, _, _ in samples
]

print("Number of samples:", len(samples))
print("Pauli error weights:", weights)
print("Mean Pauli error weight:", np.mean(weights))

Number of samples: 15
Pauli error weights: [3, 4, 4, 8, 1, 3, 2, 7, 3, 6, 7, 6, 4, 8, 2]
Mean Pauli error weight: 4.533333333333333


In [18]:
results = []

# starting at 1 rather than 0 for the sake of presentation
for trial, (e_X_trial, e_Z_trial, s_X_trial, s_Z_trial) in enumerate(samples, start=1):
    # decode the X-error component
    estimate_X, consistent_X, iterations_X = decode_binary(
        H_Z, s_Z_trial, p=p_trials, max_iter=20
    )

    # decode the Z-error component
    estimate_Z, consistent_Z, iterations_Z = decode_binary(
        H_X, s_X_trial, p=p_trials, max_iter=20
    )

    exact_recovery = np.array_equal(
        np.asarray(estimate_X), e_X_trial
    ) and np.array_equal(np.asarray(estimate_Z), e_Z_trial)

    # check logical decoding success
    success = logical_success(H_X, H_Z, e_X_trial, e_Z_trial, estimate_X, estimate_Z)

    results.append(
        {
            "Trial": trial,
            "Pauli weight": int(np.count_nonzero(e_X_trial | e_Z_trial)),
            "X consistent": bool(consistent_X),
            "Z consistent": bool(consistent_Z),
            "X iterations": iterations_X,
            "Z iterations": iterations_Z,
            "Exact recovery": bool(exact_recovery),
            "Logical success": bool(success),
        }
    )

print("Number of decoded samples:", len(results))

Number of decoded samples: 15


In [19]:
num_trials = len(results)

# Count exact recoveries.
num_exact = sum(row["Exact recovery"] for row in results)

# Count logical decoding successes.
num_logical = sum(row["Logical success"] for row in results)

# Count trials where both CSS components are syndrome-consistent.
num_consistent = sum(row["X consistent"] and row["Z consistent"] for row in results)

print("Total trials:", num_trials)
print("Syndrome-consistent trials:", num_consistent)
print("Exact recoveries:", num_exact)
print("Logical successes:", num_logical)

assert num_exact <= num_logical <= num_consistent <= num_trials

Total trials: 15
Syndrome-consistent trials: 11
Exact recoveries: 10
Logical successes: 11


In [20]:
columns = [
    "Trial",
    "Pauli weight",
    "X iterations",
    "Z iterations",
    "X consistent",
    "Z consistent",
    "Exact recovery",
    "Logical success",
]

# print column headings
print(" | ".join(f"{name:>16}" for name in columns))
print("-" * 149)

# print the values for each trial
for row in results:
    print(" | ".join(f"{row[name]!s:>16}" for name in columns))


for row in results:
    if row["Logical success"] and not row["Exact recovery"]:
        print(row)

           Trial |     Pauli weight |     X iterations |     Z iterations |     X consistent |     Z consistent |   Exact recovery |  Logical success
-----------------------------------------------------------------------------------------------------------------------------------------------------
               1 |                3 |                1 |                1 |             True |             True |             True |             True
               2 |                4 |               20 |                1 |            False |             True |            False |            False
               3 |                4 |                2 |                2 |             True |             True |             True |             True
               4 |                8 |                2 |               20 |             True |            False |            False |            False
               5 |                1 |                1 |                1 |             True |      

### Stabilizer Equivalence in Trial 11

Among the 15 samples, Trial 11 is the only case in which BP achieved logical decoding success without exact recovery. To examine this distinction directly, we reconstruct its residual error and test whether its X and Z components belong to their respective stabilizer row spaces.

In [21]:
# retrieve trial 11
e_X_11, e_Z_11, s_X_11, s_Z_11 = samples[10]

# repeat the decoding of this particular sample
estimate_X_11, _, _ = decode_binary(H_Z, s_Z_11, p=p_trials, max_iter=20)

estimate_Z_11, _, _ = decode_binary(H_X, s_X_11, p=p_trials, max_iter=20)


r_X_11 = (e_X_11 + np.asarray(estimate_X_11)) % 2
r_Z_11 = (e_Z_11 + np.asarray(estimate_Z_11)) % 2

residual_weight = np.count_nonzero(r_X_11 | r_Z_11)

print("Residual Pauli weight:", residual_weight)


x_stabilizer = is_in_row_space(H_X, r_X_11)
z_stabilizer = is_in_row_space(H_Z, r_Z_11)

print("X residual is a stabilizer:", x_stabilizer)
print("Z residual is a stabilizer:", z_stabilizer)
print("X residual weight:", np.count_nonzero(r_X_11))
print("Z residual weight:", np.count_nonzero(r_Z_11))

Residual Pauli weight: 6
X residual is a stabilizer: True
Z residual is a stabilizer: True
X residual weight: 6
Z residual weight: 0


### Discussion

We tested our BP decoder on 15 independently sampled Pauli errors at physical error probability $p=0.06$. The decoder produced syndrome-consistent estimates in 11 trials, with exact recovery in 10 and logical decoding success in 11. In Trial 11, the estimated error differed from the actual error by a weight-six X-type stabilizer, illustrating that exact recovery is not necessary for successful quantum decoding.

In the remaining four trials, at least one binary decoder reached the maximum of 20 iterations without finding a syndrome-consistent estimate. These are *heralded failures*, in the terminology of Higgott and Breuckmann [1, Sec. V]. Although every syndrome-consistent estimate in this experiment was logically successful, BP can also return a syndrome-consistent correction that leaves a nontrivial logical error.

The observed logical failure fraction was $4/15$. More generally, the *logical failure rate* at a fixed physical error probability $p$ is the probability that the residual error after recovery is not a stabilizer, encompassing both syndrome-inconsistent corrections and undetected logical errors. Our sample is too small to estimate this probability reliably.

OSD addresses heralded BP failures by using the final soft decisions to construct syndrome-consistent corrections [1, Sec. VI]. We will implement this postprocessing step next and compare BP against BP+OSD on the BB code.

## References

1. O. Higgott and N. P. Breuckmann, "Improved Single-Shot Decoding of Higher-Dimensional Hypergraph-Product Codes," *PRX Quantum* **4**, 020332 (2023). https://doi.org/10.1103/PRXQuantum.4.020332

2. T. Ginsberg, "Decoding Quantum Errors on the Steane Code with Belief Propagation & Catalyst," *PennyLane Demos* (2025, updated 2026). https://pennylane.ai/demos/tutorial_bp_catalyst

3. A. Rigby, J. C. Olivier, and P. Jarvis, "Modified belief propagation decoders for quantum low-density parity-check codes," *Physical Review A* **100**, 012330 (2019). https://doi.org/10.1103/PhysRevA.100.012330

4. S. Bravyi, A. W. Cross, J. M. Gambetta, D. Maslov, P. Rall, and T. J. Yoder, "High-threshold and low-overhead fault-tolerant quantum memory," *Nature* **627**, 778–782 (2024). https://doi.org/10.1038/s41586-024-07107-7
